In [1]:
# Import basic libraries and setup environment
import os
import joblib
import numpy as np
import pandas as pd # Required for dataframe operations like select_dtypes

# Import machine learning models
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Create folder to save trained models and predictions
os.makedirs("checkpoints", exist_ok=True)
print("Folder 'checkpoints' is ready!")

Folder 'checkpoints' is ready!


In [12]:
# Load the balanced dataset
#1. Load objects from a .npz file.
data_bal = np.load("balanced_train.npz", allow_pickle=True)

# Print the keys currently in the file for easy cross-checking
print("Variables found:", list(data_bal.keys()))

# 2. Extract data from the data_bal container
X_train_lgb_bal = data_bal['X_train_lgb_bal']
y_train_bal = data_bal['y_train_bal']
X_train_cat_bal_raw = data_bal['X_train_cat_bal']

# 3. Check if cat_features is already a list or numpy array
cat_features_data = data_bal['cat_features']
if isinstance(cat_features_data, np.ndarray):
    cat_features_list = cat_features_data.tolist()
else:
    cat_features_list = list(cat_features_data) # If it's already a list

# 4. Convert to DataFrame (Essential for CatBoost)
X_train_cat_bal = pd.DataFrame(X_train_cat_bal_raw)

print(f"Data loaded! Categorical columns: {cat_features_list}")

Variables found: ['X_train_lgb_bal', 'X_train_cat_bal', 'y_train_bal', 'X_test_lgb', 'X_test_cat', 'y_test', 'cat_features']
Data loaded! Categorical columns: ['Cluster']


In [13]:
# Train LightGBM Model
# LightGBM model initialization (works best with encoded numerical features)
lgb_model = LGBMClassifier(
    n_estimators=500,     # Total number of trees
    learning_rate=0.05,   # Step size for each iteration
    max_depth=-1,         # No limit for tree depth (Leaf-wise growth)
    random_state=42,      # Ensure reproducible results
    n_jobs=-1             # Use all available CPU cores for faster training
)

# Train LightGBM on the balanced dataset
print("Training LightGBM...")
lgb_model.fit(X_train_lgb_bal, y_train_bal)
print("LightGBM Training Completed!")

# Save the trained model to the checkpoints folder
joblib.dump(lgb_model, "checkpoints/lightgbm_model.pkl")
print("LightGBM model saved successfully!")

Training LightGBM...
[LightGBM] [Info] Number of positive: 740, number of negative: 740
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002190 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1578
[LightGBM] [Info] Number of data points in the train set: 1480, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning]

In [14]:
# Preprocess data for CatBoost
# Create a copy to avoid changing the original training data
X_cat = X_train_cat_bal.copy()

# Identify all categorical columns automatically
cat_features = X_cat.select_dtypes(include=['object', 'category']).columns.tolist()
print("Categorical features detected:", cat_features)

# Process categorical columns to fix potential SMOTE decimal issues
for col in cat_features:
    X_cat[col] = X_cat[col].astype('object')     # Convert to object first to handle mixed types safely
    X_cat[col] = X_cat[col].fillna("Missing")    # Fill any missing values with a string placeholder
    X_cat[col] = X_cat[col].astype(str)          # Ensure all values are strictly strings for CatBoost

print("Categorical features preprocessed successfully!")

Categorical features detected: ['Cluster']
Categorical features preprocessed successfully!


In [15]:
# Train CatBoost Model
# CatBoost model initialization (handles categorical features directly)
cat_model = CatBoostClassifier(
    iterations=500,       # Total number of trees
    learning_rate=0.05,   # Step size
    depth=6,              # Depth of the symmetric trees
    random_seed=42,       # Ensure reproducible results
    verbose=100           # Print training log every 100 iterations to track progress
)

# Train CatBoost
print("Training CatBoost...")
cat_model.fit(
    X_cat,
    y_train_bal,
    cat_features=cat_features # Explicitly define which columns are categorical
)
print("CatBoost Training Completed!")

# Save the trained model
joblib.dump(cat_model, "checkpoints/catboost_model.pkl")
print("CatBoost model saved successfully!")

Training CatBoost...
0:	learn: 0.5566822	total: 186ms	remaining: 1m 32s
100:	learn: 0.0007131	total: 3.5s	remaining: 13.8s
200:	learn: 0.0006292	total: 6.65s	remaining: 9.89s
300:	learn: 0.0006277	total: 9.8s	remaining: 6.48s
400:	learn: 0.0006264	total: 12.9s	remaining: 3.2s
499:	learn: 0.0006254	total: 16.1s	remaining: 0us
CatBoost Training Completed!
CatBoost model saved successfully!


In [16]:
# Generate predictions for model comparison
# Predict probability (class 1 = churn)
print("Generating probability predictions...")
lgb_pred = lgb_model.predict_proba(X_train_lgb_bal)[:, 1]
cat_pred = cat_model.predict_proba(X_cat)[:, 1]

# Save predictions
joblib.dump(lgb_pred, "checkpoints/lgb_train_pred.pkl")
joblib.dump(cat_pred, "checkpoints/cat_train_pred.pkl")
print("Predictions saved successfully! Ready for comparative evaluation.")

Generating probability predictions...
Predictions saved successfully! Ready for comparative evaluation.


In [17]:
# Check generated files
import os

files_to_check = [
    "checkpoints/lightgbm_model.pkl",
    "checkpoints/catboost_model.pkl",
    "checkpoints/lgb_train_pred.pkl",
    "checkpoints/cat_train_pred.pkl"
]

print("Check folder 'checkpoints':")
for f in files_to_check:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024 # KB
        print(f"{f} - Size: {size:.2f} KB")
    else:
        print(f"{f} is MISSING!")

Check folder 'checkpoints':
checkpoints/lightgbm_model.pkl - Size: 164.02 KB
checkpoints/catboost_model.pkl - Size: 546.88 KB
checkpoints/lgb_train_pred.pkl - Size: 11.78 KB
checkpoints/cat_train_pred.pkl - Size: 11.78 KB
